In [ ]:
import pandas as pd
import numpy as np
import sys
import os 


sys.path.append(os.path.abspath("../../")) 


from sklearn import set_config

from sklearn.model_selection import train_test_split
from sksurv.util import Surv
set_config(display="text")  # displays text representation of estimators


from src.utils.ConvertTextToCsv import TextToCsv
from src.dataset.generate_dataset import TorchPreprocessing
from sksurv.linear_model import CoxnetSurvivalAnalysis
from src.utils.Preprocessing import Preprocessor
from sksurv.ensemble.forest import RandomSurvivalForest
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival
from src.utils.cox_models import *
from src.utils.kapplan_meir import k_m_surv
from src.utils.set_seed import set_seed
from src.utils.minimal_depth_interactions import minimal_depth, interaction_matrix, top_interacting_pairs

import json
import torch
import random
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
set_seed(42)

In [3]:
pp = Preprocessor()

In [4]:
set_seed()
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")


Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 299 - Total(%) 0.37
Shape of the CSV: (20440, 819)


In [5]:
total_columns = len(df_mRNA_raw_data) - 50
total_columns

20390

In [6]:
df_merged = TorchPreprocessing(df_mRNA_raw_data,df_clinical_data, total_columns).get_comparation_df()
 

Genes before Treshold: 20393
count    20393.000000
mean        74.057471
std        155.640823
min          0.000000
25%          0.000000
50%          0.000000
75%         21.000000
max        519.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 18492


In [7]:
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
]

In [8]:
status = comparation_df["Overall Survival Status"].astype(str).str.strip()
comparation_df["event"] = status.str.contains("DECEASED", na=False)
comparation_df["event_60"] = comparation_df["event"].copy()
comparation_df["time_60"] = np.minimum(comparation_df["Overall Survival (Months)"], 60)
comparation_df.loc[
    comparation_df["Overall Survival (Months)"] > 60, "event_60"
] = False
comparation_df = comparation_df.dropna(subset=["time_60"]).copy()


In [9]:
# splits.json, so the NN notebook reuses the SAME patients.
# ---- Canonical split BY Sample ID, saved once so every model uses the SAME patients ----
SPLITS_PATH = "splits.json"
if not os.path.exists(SPLITS_PATH):
    ids = comparation_df["Sample ID"].astype(str).to_numpy()
    ev = comparation_df["event_60"].to_numpy()
    trainval_ids, test_ids = train_test_split(ids, test_size=104, stratify=ev, random_state=42)
    ev_tv = comparation_df.set_index("Sample ID").loc[trainval_ids, "event_60"].to_numpy()
    train_ids, val_ids = train_test_split(trainval_ids, test_size=84, stratify=ev_tv, random_state=42)
    json.dump(
        {"train_ids": train_ids.tolist(),
         "val_ids": val_ids.tolist(),
         "test_ids": test_ids.tolist()},
        open(SPLITS_PATH, "w"), indent=2,
    )


In [10]:
split = json.load(open(SPLITS_PATH))
trainval_ids = split["train_ids"] + split["val_ids"]   # classic models train on 415 (CV inside)
test_ids = split["test_ids"]                            # shared 104 test patients

In [11]:
sid = comparation_df["Sample ID"].astype(str)
train_idx = comparation_df.index[sid.isin(trainval_ids)]
test_idx = comparation_df.index[sid.isin(test_ids)]
df_train = comparation_df.loc[train_idx]
df_test = comparation_df.loc[test_idx]
print(f"Train+Val: {len(train_idx)}, Test: {len(test_idx)}")

Train+Val: 415, Test: 104


In [12]:

results_df, desing, expr = pp.initialize_limma(df_train.drop(columns=["event", "event_60", "time_60"]),column="Tumor-Cancer",column_event="Overall Survival (Months)", column_status="Overall Survival Status",
)

In [13]:
# Univariate Cox screening, also on the TRAINING split only.
set_seed()
N_GENES = 1000
top_genes_limma = results_df.sort_values("pvalue").index[:N_GENES]
X_candidates = df_train[top_genes_limma].apply(pd.to_numeric, errors='coerce')

from lifelines import CoxPHFitter

cox_uni_results = []

for gene in top_genes_limma:
    try:
        df_gene = pd.DataFrame({
            'gene': X_candidates[gene],
            'time': df_train['time_60'],
            'event': df_train['event_60']
        }).dropna()

        cph = CoxPHFitter()
        cph.fit(df_gene, duration_col='time', event_col='event')

        cox_uni_results.append({
            'gene': gene,
            'beta': cph.params_['gene'],
            'p_value': cph.summary['p']['gene'],
            'log_rank_p': cph.log_likelihood_ratio_test().p_value
        })
    except Exception:
        continue

cox_uni_df = pd.DataFrame(cox_uni_results).set_index('gene')
cox_uni_df = cox_uni_df.sort_values('p_value')

top_genes = cox_uni_df.head(1000).index

In [14]:
X = comparation_df[top_genes].apply(pd.to_numeric, errors='coerce')
X = X.fillna(0).replace([np.inf, -np.inf], 0)

Y = Surv.from_dataframe(event="event_60", time="time_60", data=comparation_df)

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
Y_train = Surv.from_dataframe(event="event_60", time="time_60", data=df_train)
Y_test = Surv.from_dataframe(event="event_60", time="time_60", data=df_test)

In [15]:
# Standardize features so the Coxnet L1/L2 penalty is applied fairly across genes.
# Fit on train only, transform test (no leakage). Both Lasso and Elastic Net use these.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index, columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index, columns=X_test.columns
)
print(f"X_train_scaled: {X_train_scaled.shape}, X_test_scaled: {X_test_scaled.shape}")

X_train_scaled: (415, 1000), X_test_scaled: (104, 1000)


In [16]:

random_forest_model = RandomSurvivalForest(
    n_estimators=128,
    min_samples_split=5,
    min_samples_leaf=16,
    random_state=42
)
random_forest_model.fit(X_train, Y_train)


RandomSurvivalForest(min_samples_leaf=16, min_samples_split=5, n_estimators=128,
                     random_state=42)

In [30]:
 # nombres de las columnas de X_train (si es DataFrame)
feature_names = list(X_train.columns)

    # paso 1: minimal depth univariante -> filtra los genes mas predictivos
md = minimal_depth(random_forest_model, feature_names)
print("Top genes por minimal depth:")
print(md.head(15))
top_genes = md.head(30).index.tolist()   # ajusta k a tu caso

    # paso 2: interacciones SOLO entre esos genes (mucho mas barato que p x p)
inter = interaction_matrix(random_forest_model, feature_names, subset=top_genes)

    # paso 3: pares que mas interaccionan
print("\nPares de genes que mas interaccionan:")
top_interacting_pairs_csv = top_interacting_pairs(inter, n=2000)
top_interacting_pairs_csv.to_csv("top_interacting_pairs.csv")

Top genes por minimal depth:
PPFIA3     5.406250
SEMA3B     5.414062
ATG3       5.468750
RANGAP1    5.468750
FAM86B1    5.476562
SBSPON     5.492188
RAP2A      5.492188
KLRG2      5.500000
ELMO3      5.507812
ATHL1      5.515625
TOMM7      5.515625
ATP2C1     5.523438
RBM4B      5.523438
CEPT1      5.531250
MDM2       5.531250
dtype: float64

Pares de genes que mas interaccionan:
